In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV

df = pd.read_csv("../data/drug_discovery_virtual_screening.csv")
df.head(3)

,compound_id,protein_id,molecular_weight,logp,h_bond_donors,h_bond_acceptors,rotatable_bonds,polar_surface_area,compound_clogp,protein_length,protein_pi,hydrophobicity,binding_site_size,mw_ratio,logp_pi_interaction,binding_affinity,active
0,CID_00000,PID_361,499.671415,2.487233,1,7,4,113.350817,4.050696,678,6.019657,0.812534,12.512165,0.736978,14.972288,5.996665,0
1,CID_00001,PID_165,436.173570,3.283222,3,4,4,71.981132,3.704408,876,6.447408,0.651417,11.538420,0.497915,21.168271,6.445742,0
2,CID_00002,PID_168,514.768854,NaN,2,11,11,83.936307,1.869610,658,3.925837,0.633467,13.155702,0.782323,9.074061,5.689583,0


In [2]:
X = df.drop(columns=["active", "compound_id", "protein_id", "binding_affinity"])
y = df["active"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)



print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 1600
Testing samples: 400


In [3]:
r2DF = pd.DataFrame(columns=["Learning Rate", "Max Depth", "Boosting Count", "R2"])
learning_rate = np.logspace(0, 1, num=10)
maxDepth = np.arange(3, 15, 1)
estimators = np.arange(3, 50, 1)

hyperparameters = {
    "learning_rate": learning_rate,
    "max_depth": maxDepth,
    "n_estimators": estimators
}

preprocesserTransformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler",  StandardScaler())
])

preprocesserTransformer.fit_transform(X_train)
X_train = preprocesserTransformer.transform(X_train)
X_test = preprocesserTransformer.transform(X_test)

In [6]:
gradBoostClassifier = GradientBoostingClassifier(random_state=42)

gradBoostGrid = GridSearchCV(
    gradBoostClassifier,
    param_grid=hyperparameters,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)
gradBoostGrid.fit(X_train, y_train)
#gradBoostClassifierBest = gradBoostClassifier.best_estimator_
print(gradBoostGrid.best_params_)

{'learning_rate': np.float64(1.0), 'max_depth': np.int64(3), 'n_estimators': np.int64(3)}


In [9]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
print("Best Validation Score:", gradBoostGrid.best_score_)
print("Best Test Score:", gradBoostGrid.best_estimator_.score(X_test, y_test))
y_pred = gradBoostGrid.predict(X_test)
y_prob = gradBoostGrid.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Gradient Boosting Accuracy:", accuracy)
print("Gradient Boosting ROC-AUC:", roc_auc)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Best Validation Score: 0.9383994292815627
Best Test Score: 0.88
Gradient Boosting Accuracy: 0.88
Gradient Boosting ROC-AUC: 0.9285587923104139

Confusion Matrix:
[[265  13]
 [ 35  87]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.95      0.92       278
           1       0.87      0.71      0.78       122

    accuracy                           0.88       400
   macro avg       0.88      0.83      0.85       400
weighted avg       0.88      0.88      0.88       400

